<a href="https://colab.research.google.com/github/redinbluesky/nlp-with-transformers/blob/main/08-효율적인_트랜스포머_구축.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  목차
* [Chapter 0 개요](#chapter0)
* [Chapter 1 의도 탐지 예제](#chapter1)
* [Chapter 2 벤치마크 클래스 만들기](#chapter2)

## Chapter 0 개요 <a class="anchor" id="chapter0"></a>

1. 모델이 너무 느리거나 크다면, 최고 성능의 모델이라도 유용하지 않다.
    - 빠르고 작은 모델을 구축하는 방법이 있다.
    - 모델의 용량을 줄이면 종종 성능이 저하되는데 이를 최소화하는 방법이 있다.
    - 지식정체, 양자화, 가지치기, ONNX포맷, ONNX런타임을 사용한 그래프최적화가 있다.

2. 로블록스 엔지니어링 팀은 블로그에 이러한 기법을 사용하여 대규모 트랜스포머 모델을 최적화하는 방법을 설명했다.
    -  지식 정제와 양자화를 연결해 레이턴시와 BERT 분류기의 성능을 30배 향상시켰다.

        ![Optimizing Large Transformer Models at Roblox](image/08-01-optimizing-large-transformer-models-at-roblox.png)

3. 각 기술의 장점과 단점을 이해하기 위해 의도 참지 예제를 사용하여 이러한 기법을 실험해본다.


## Chapter 1 의도 탐지 예제 <a class="anchor" id="chapter0"></a>
1. 고객이 다음과 같은 메시지를 보냈다고 가정한다.
    - "Hey, I'd like to rent a vehicle from Nov 1st to Nov 15th In Paris and I need a 15 passanger van."

2. 의도 분류기는 이를 자동으로 Car Rent로 분류하고 응답한다.
    - 고객이 사전에 저으이된 의도에 속하지 않은 쿼리를 제공하면 시스템은 대체 응답을 출력해댜한다.
    - 아래의 그림은 범위 안에 없는 질문을 하자 잘 못된 응답을 출력했고, 세 번재 질문에서 카테고리에 없는 응답이라는 것을 인지하고 적절한 응답을 출력했다.

        ![Intent Detection Example](image/08-02-intent-detection-example.png)

3. CLINC150 데이터셋에서 미세튜닝해 약 94%의 정확도를 달성한 BERT 베이스 모델을 기준모델로 사용한다.
    - 이 데이터셋에는 150개의 의도와 은행, 여행 등 10개 분야로 분류된 22,500개 쿼리가 포함되어 있다.
    - 범위를 벗어난 쿼리가 1,200개 있고 이런 쿼리는 의도 클래스 oos에 속한다.
    - 실전에서는 회사 내부 데이터셋도 수집하겟지만, 공개 데이터를 사용하는 것이 빠르게 반복하고 초기 결과를 생성하기 좋다.

In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [2]:
from transformers import pipeline

bert_ckpt = "transformersbook/bert-base-uncased-finetuned-clinc"
pipe = pipeline("text-classification", model=bert_ckpt)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: transformersbook/bert-base-uncased-finetuned-clinc
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# 쿼리를 전달해 모델로부터 예측한 의도와 신뢰점수를 얻는다.
query = "Hey, I'd like to rent a vehicle from Nov 1st to Nov 15th In Paris and I need a 15 passanger van."
pipe(query)

[{'label': 'car_rental', 'score': 0.5804867148399353}]

## Chapter 2 벤치마크 클래스 만들기 <a class="anchor" id="chapter2"></a>
1. 애플리케이션의 중요 지표가 이미정의되어 있다고 가정하고 모델의 지표를 최적화하는 데 집중한다.

2. 모델 성능이 중요한 경우는 다음과 같다.
   - 오류가 발생했을 때 손실 비용이 큰 상황(또는 오류 발생을 줄이기 위해 사람의 참여가 최선일 때)
   - 모델의 지표가 조금 향상되면 전체적으로 큰 이득을 얻을 수 있는 상황 

3. 레이턴시가 중요한 경우
    - 대량의 트래픽을 처리하는 실시간 환경에서 고려한다.

4. 메모리가 중요한 경우
    - 메모리가 모바일과 에지 장치에서 제한적일 때 고려한다.
    

In [4]:
# 파이프라인과 테스트 세트가 주어지면 성능을 측정하는 간단한 벤치마크 클래스를 만든다.
class PerformanceBenchmark:
    """
    optim_type: str
        여러가지 최적화 기법의 성능을 비교하기 위한 문자열 식별자
    """
    def __init__(self, pipeline, dataset, optim_type="BERT baseline"):
        self.pipeline = pipeline
        self.dataset = dataset
        self.optim_type = optim_type

    def compute_accuracy(self):
        # 나중에 정의한다.
        pass

    def compute_size(self):
        # 나중에 정의한다.
        pass

    def time_pipeline(self):
        # 나중에 정의한다.
        pass

    def run_benchmark(self):
        """딕셔너리에 optim_type을 키로 하여 성능 메트릭을 저장한다."""
        metrics = {}
        metrics[self.optim_type] = self.compute_size()
        metrics[self.optim_type].update(self.time_pipeline())
        metrics[self.optim_type].update(self.compute_accuracy())
        return metrics

In [ ]:
# 기준 모델을 미세 튜닝하는 데 사용한 CLINC150 데이터셋을 불러온다.
from datasets import load_dataset

# "plus": 범위 밖의 훈ㅁ련 샘플이 담긴 서브셋을 의미한다.
# "clinc_oos" 데이터셋에는 150개의 의도 클래스가 있다.
clinc = load_dataset("clinc_oos", "plus")

README.md: 0.00B [00:00, ?B/s]

plus/train-00000-of-00001.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

plus/validation-00000-of-00001.parquet:   0%|          | 0.00/77.8k [00:00<?, ?B/s]

plus/test-00000-of-00001.parquet:   0%|          | 0.00/136k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15250 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5500 [00:00<?, ? examples/s]

In [ ]:
# 테스트 샘플 하나를 살표본다.
sample = clinc["test"][42]
print(f"샘플 쿼리: {sample}")

샘플 쿼리: {'text': 'transfer $100 from my checking to saving account', 'intent': 133}


In [11]:
# 의도는 ID로 제공되지만 features 속성을 사용하면 문자열로 매핑된다.
intents = clinc["test"].features["intent"]
print(f"샘플 의도 ID: {sample['intent']}, 의도 이름: {intents.int2str(sample['intent'])}")

샘플 의도 ID: 133, 의도 이름: transfer


In [ ]:
# commute_accuracy() 메서드를 구현한다.
import evaluate

accuracy_score = evaluate.load("accuracy")

In [ ]:
def compute_accuracy(self):
    """
    정확한 정수 지표는 예측과 정답을 기대한다.
    파이프라인을 사용해 text 필드에서 예측을 추출하고, intent 객체의 str2int()를 사용해 각 예측을 해당 ID로 변환한다.
    """
    preds, labels = [], []
    for example in self.dataset:
        pred = self.pipeline(example["text"])[0]["label"]
        label = example["intent"]
        preds.append(intents.str2int(pred))
        labels.append(label)
    accuracy = accuracy_score.compute(predictions=preds, references=labels)
    print(f"정확도: {accuracy['accuracy']:.3f}")
    return accuracy

PerformanceBenchmark.compute_accuracy = compute_accuracy

5. 파이토치의 torch.save() 함수를 사용해 모델을 디스크에 직렬화하고 크기를 계산한다.
    - 파이토치에서 모델을 저장할 때 state_dict() 메서드를 사용해 모델의 매개변수를 저장한다.
    - 각각의 키/값 쌍이 BERT의 층과 텍션에 해당한다.

In [17]:
import torch

list(pipe.model.state_dict().items())[42]
torch.save(pipe.model.state_dict(), "model.pt")

In [ ]:
import torch
from pathlib import Path

def compute_size(self):
    """
    Path.stat() 메서드를 사용해 모델 파일의 크기를 바이트 단위로 계산한다.
    """
    state_dic = self.pipeline.model.state_dict()
    tmp_path = Path("model.pt")
    torch.save(state_dic, tmp_path)
    # 메가바이트 단위로 크기를 계산한다.
    size_mb = tmp_path.stat().st_size / (1024 * 1024)
    print(f"모델 크기: {size_mb:.2f} MB")
    # 임시 파일을 삭제한다.
    tmp_path.unlink()
    return {"size_mb": size_mb}

PerformanceBenchmark.compute_size = compute_size